# Slide 40 – Chapter 5: Phân tích độ tương đồng giữa các bài báo VnExpress

## Đề bài (Bài tập nhóm – slide 40)
- Crawl phần văn bản của các chuyên mục tin tức sau, **mỗi chuyên mục 2 bài mới nhất**:
  - News1: https://vnexpress.net/rss/the-thao.rss (Thể thao)
  - News2: https://vnexpress.net/rss/suc-khoe.rss (Sức khỏe)
  - News3: https://vnexpress.net/rss/kinh-doanh.rss (Kinh doanh)
- Lưu file theo cú pháp: `News11.txt, News12.txt, …, News21.txt, News22.txt, …`
- Đánh giá mức độ tương đồng giữa các trang web.
- Sắp xếp các cặp tin tức theo độ tương đồng **giảm dần**.
- **Yêu cầu:** code được tích hợp, chạy 1 lần thực hiện liên tục tất cả các bước và in kết quả lên màn hình.

## Hướng làm: dữ liệu *realtime*
Không dùng file có sẵn: **mỗi lần chạy** notebook sẽ đọc RSS tại thời điểm đó → lấy 2 bài mới nhất → crawl → tính toán.
Vì vậy kết quả (bài báo, điểm số) có thể khác nhau giữa các lần chạy.

## Quy trình 5 bước (theo slide 37–38)
| Bước | Nội dung | Công cụ |
|---|---|---|
| 1 | Thu thập dữ liệu: đọc RSS, crawl nội dung bài | `requests`, `xml.etree`, `BeautifulSoup` |
| 2 | Tiền xử lý: chữ thường, bỏ ký tự đặc biệt, tách từ, bỏ stopwords | `re`, `pyvi` |
| 3 | Xây dựng vector đặc trưng | `TfidfVectorizer` / `CountVectorizer` (BOW) |
| 4 | Tính độ tương đồng | `cosine_similarity` |
| 5 | Kết luận: xếp hạng các cặp theo độ tương đồng giảm dần | `pandas` |

## 0. Cài đặt thư viện (chỉ cần chạy 1 lần)
- `requests`: gửi HTTP request để tải RSS và trang bài báo.
- `beautifulsoup4`: phân tích HTML, lấy đúng phần nội dung bài.
- `pyvi`: tách từ tiếng Việt (ghép từ nhiều âm tiết, ví dụ "bóng đá" → `bóng_đá`).
- `scikit-learn`: vector hoá văn bản (TF-IDF, BOW) và tính Cosine.
- `pandas`: hiển thị ma trận và bảng xếp hạng.

Bỏ dấu `#` ở dòng dưới nếu máy chưa cài.

In [ ]:
# !pip install requests beautifulsoup4 pyvi scikit-learn pandas

## 1. Import thư viện và cấu hình
- `RSS_FEEDS`: ánh xạ **số chuyên mục** → (tên, link RSS). Số này chính là chữ số thứ nhất trong tên file `News{i}{j}.txt`.
- `ARTICLES_PER_FEED = 2`: mỗi chuyên mục lấy 2 bài (chữ số thứ hai `j` = 1, 2).
- `HEADERS`: giả lập trình duyệt (User-Agent) để server không chặn request từ Python.
- File `vietnamese-stopwords.txt` phải nằm cùng thư mục với notebook.

In [ ]:
import os                                   # làm việc với đường dẫn, thư mục
import re                                   # biểu thức chính quy (regex) để làm sạch văn bản
import time                                 # sleep khi cần thử lại request
import xml.etree.ElementTree as ET          # đọc file RSS (định dạng XML) - có sẵn trong Python
from datetime import datetime               # in thời điểm chạy
from email.utils import parsedate_to_datetime  # đổi chuỗi ngày RSS "Thu, 22 Oct 2026 15:14:19 +0700" -> datetime
from itertools import combinations          # sinh tất cả các cặp (i, j) với i < j

import pandas as pd                         # hiển thị bảng
import requests                             # tải trang web
from bs4 import BeautifulSoup               # phân tích HTML
from pyvi import ViTokenizer                # tách từ tiếng Việt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # BOW / TF-IDF
from sklearn.metrics.pairwise import cosine_similarity                        # độ đo Cosine

BASE_DIR = os.getcwd()                                  # thư mục đang chạy notebook
DATA_DIR = os.path.join(BASE_DIR, "data")               # nơi lưu News11.txt, News12.txt, ...
STOPWORDS_FILE = os.path.join(BASE_DIR, "vietnamese-stopwords.txt")

# Số chuyên mục -> (tên chuyên mục, link RSS)
RSS_FEEDS = {
    1: ("Thể thao", "https://vnexpress.net/rss/the-thao.rss"),
    2: ("Sức khỏe", "https://vnexpress.net/rss/suc-khoe.rss"),
    3: ("Kinh doanh", "https://vnexpress.net/rss/kinh-doanh.rss"),
}
ARTICLES_PER_FEED = 2   # số bài mới nhất cần lấy cho mỗi chuyên mục

# Giả lập trình duyệt Chrome để tránh bị server từ chối
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"}

pd.set_option("display.width", 200)   # cho bảng pandas hiển thị rộng, không bị xuống dòng

## 2. Bước 1 – Thu thập dữ liệu (crawl)

### 2.1. Hàm `get(url)`: tải trang có thử lại
Mạng có thể lỗi tạm thời (timeout, mất kết nối) → thử lại tối đa 3 lần, mỗi lần cách nhau 2 giây.
`resp.encoding = "utf-8"` để tiếng Việt có dấu không bị lỗi font.

In [ ]:
def get(url, retries=3):
    # Tải nội dung (HTML/XML) của url, thử lại nếu bị lỗi mạng.
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)  # chờ tối đa 15 giây
            resp.raise_for_status()      # nếu mã HTTP là 4xx/5xx thì ném lỗi
            resp.encoding = "utf-8"      # VnExpress dùng UTF-8
            return resp.text
        except requests.RequestException as e:
            if attempt == retries - 1:   # đã thử hết số lần -> báo lỗi ra ngoài
                raise
            print(f"   ! Lỗi {e}, thử lại...")
            time.sleep(2)

### 2.2. Hàm `fetch_rss(url)`: đọc RSS, lấy danh sách bài mới nhất
RSS là file XML, mỗi bài nằm trong thẻ `<item>`:
```xml
<item>
  <title>Fabregas vượt trội Carrick thế nào về chiến thuật</title>
  <pubDate>Thu, 22 Oct 2026 15:14:19 +0700</pubDate>
  <link>https://vnexpress.net/fabregas-vuot-troi-...-5135624.html</link>
</item>
```
Hàm lấy `title`, `link`, `pubDate` của từng bài rồi **sắp xếp theo thời gian đăng giảm dần**,
để chắc chắn phần tử đầu danh sách là bài mới nhất (không phụ thuộc vào thứ tự trong file RSS).

In [ ]:
def fetch_rss(url):
    # Trả về danh sách bài [{title, link, pub_date}] đã sắp xếp: mới nhất đứng trước.
    root = ET.fromstring(get(url).encode("utf-8"))   # parse chuỗi XML thành cây
    items = []
    for item in root.iter("item"):                   # duyệt mọi thẻ <item>
        items.append({
            "title": item.findtext("title", "").strip(),
            "link": item.findtext("link", "").strip(),
            # đổi chuỗi ngày dạng RFC 822 sang datetime để so sánh được
            "pub_date": parsedate_to_datetime(item.findtext("pubDate").strip()),
        })
    return sorted(items, key=lambda x: x["pub_date"], reverse=True)

### 2.3. Hàm `crawl_article(url)`: lấy nội dung văn bản của 1 bài
Cấu trúc HTML trang bài viết VnExpress (xem bằng F12 → Inspect):
| Thành phần | Selector CSS |
|---|---|
| Tiêu đề | `h1.title-detail` |
| Sapo (đoạn tóm tắt) | `p.description` |
| Thân bài | các thẻ `p.Normal` nằm trong `article.fck_detail` |

Lưu ý:
- Đoạn cuối bài thường là **tên tác giả** (`<p class="Normal" style="text-align:right;">`) → bỏ đi vì không phải nội dung.
- Bài dạng **video / ảnh** không có `article.fck_detail` hoặc không có đoạn văn → trả về `None` để bỏ qua.

In [ ]:
def crawl_article(url):
    # Trả về văn bản = tiêu đề + sapo + thân bài; None nếu bài không có nội dung chữ.
    soup = BeautifulSoup(get(url), "html.parser")
    body = soup.select_one("article.fck_detail")         # khung chứa thân bài
    if body is None:                                      # bài video/ảnh không có khung này
        return None

    # Lấy text từng đoạn p.Normal, bỏ đoạn căn phải (tên tác giả)
    paragraphs = [p.get_text(" ", strip=True) for p in body.select("p.Normal")
                  if "text-align:right" not in (p.get("style") or "").replace(" ", "")]
    paragraphs = [p for p in paragraphs if p]             # bỏ đoạn rỗng
    if not paragraphs:
        return None

    title = soup.select_one("h1.title-detail")
    desc = soup.select_one("p.description")
    parts = [title.get_text(strip=True) if title else "",
             desc.get_text(" ", strip=True) if desc else ""] + paragraphs
    return "\n".join(x for x in parts if x)               # mỗi phần 1 dòng

### 2.4. Hàm `crawl_all()`: crawl 3 chuyên mục và lưu file `NewsXY.txt`
Với mỗi chuyên mục `i`:
1. Đọc RSS, duyệt từ bài mới nhất.
2. Crawl từng bài; bài nào không có văn bản thì **bỏ qua và lấy bài kế tiếp**, để luôn đủ 2 bài.
3. Lưu bài thứ `j` vào `data/News{i}{j}.txt` (ví dụ chuyên mục 2, bài 1 → `News21.txt`).

Hàm trả về danh sách `docs`, mỗi phần tử gồm `name`, `category`, `title`, `text`, dùng cho các bước sau.

In [ ]:
def crawl_all():
    os.makedirs(DATA_DIR, exist_ok=True)       # tạo thư mục data/ nếu chưa có
    docs = []
    for i, (cat, rss_url) in RSS_FEEDS.items():
        print(f"\n[News{i}] {cat}: {rss_url}")
        j = 0                                  # số bài đã lấy được của chuyên mục này
        for item in fetch_rss(rss_url):
            if j == ARTICLES_PER_FEED:         # đủ 2 bài -> sang chuyên mục tiếp theo
                break
            text = crawl_article(item["link"])
            if text is None:                   # bài video / ảnh, lấy bài kế tiếp
                print(f"   - Bỏ qua (không có nội dung văn bản): {item['title']}")
                continue
            j += 1
            name = f"News{i}{j}"               # News11, News12, News21, ...
            with open(os.path.join(DATA_DIR, name + ".txt"), "w", encoding="utf-8") as f:
                f.write(text)
            print(f"   + {name}.txt | {item['pub_date']:%d/%m/%Y %H:%M} | {item['title']}")
            docs.append({"name": name, "category": cat, "title": item["title"], "text": text})
    return docs

## 3. Bước 2 – Tiền xử lý văn bản
Mục đích: chỉ giữ lại các **từ mang nghĩa**, để vector đặc trưng phản ánh đúng chủ đề bài.

1. **Chuyển chữ thường**: "Man Utd" và "man utd" được coi là một.
2. **Bỏ URL, dấu câu, số, ký tự đặc biệt** bằng regex.
3. **Tách từ tiếng Việt** bằng `pyvi`: tiếng Việt viết cách từng âm tiết, nhưng một từ có thể gồm nhiều âm tiết.
   `pyvi` nối chúng bằng `_`, ví dụ `đái tháo đường` → `đái_tháo_đường`.
   Nhờ đó "đái_tháo_đường" là **1 đặc trưng**, thay vì 3 âm tiết rời rạc dễ trùng với từ khác.
4. **Bỏ stopwords** (từ dừng như "và", "của", "là", "những"…): xuất hiện ở mọi bài nên không giúp phân biệt chủ đề.
   File stopwords viết từ ghép bằng dấu cách ("ai nấy") → đổi thành `ai_nấy` cho khớp với kết quả của `pyvi`.
5. Bỏ các token chỉ có 1 ký tự.

In [ ]:
def load_stopwords():
    with open(STOPWORDS_FILE, encoding="utf-8") as f:
        # pyvi nối từ ghép bằng "_" nên chuẩn hoá stopwords theo cùng dạng
        return {line.strip().lower().replace(" ", "_") for line in f if line.strip()}


def preprocess(text, stopwords):
    text = text.lower()                              # 1. chữ thường
    text = re.sub(r"https?://\S+", " ", text)        # 2a. bỏ URL
    text = re.sub(r"[^\w\s]|\d|_", " ", text)        # 2b. bỏ dấu câu, chữ số, dấu "_"
    text = re.sub(r"\s+", " ", text).strip()         # gộp nhiều khoảng trắng thành 1
    tokens = ViTokenizer.tokenize(text).split()      # 3. tách từ tiếng Việt
    return " ".join(t for t in tokens                # 4-5. bỏ stopwords và token 1 ký tự
                    if t not in stopwords and len(t) > 1)

**Ví dụ minh hoạ** tiền xử lý trên một câu:

In [ ]:
sample = "Người bệnh đái tháo đường có nguy cơ suy thận cao hơn 2-3 lần so với người bình thường."
print("Gốc        :", sample)
print("Sau xử lý  :", preprocess(sample, load_stopwords()))

Gốc        : Người bệnh đái tháo đường có nguy cơ suy thận cao hơn 2-3 lần so với người bình thường.
Sau xử lý  : người_bệnh đái_tháo_đường nguy_cơ suy thận bình_thường


## 4. Bước 3 & 4 – Vector đặc trưng và độ tương đồng Cosine

### Vector đặc trưng
- **BOW (`CountVectorizer`)**: mỗi chiều là 1 từ trong bộ từ vựng, giá trị = **số lần** từ đó xuất hiện trong bài.
- **TF-IDF (`TfidfVectorizer`)** – mặc định dùng trong bài:
  $$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \ln\frac{1 + N}{1 + \text{df}(t)} + 1$$
  - `tf`: tần suất từ `t` trong bài `d`; `df`: số bài chứa từ `t`; `N`: tổng số bài (ở đây là 6).
  - Từ xuất hiện ở **nhiều bài** (ít đặc trưng) bị giảm trọng số; từ **riêng của một chủ đề** (ví dụ `chiến_thuật`, `thận`) được tăng trọng số.

### Độ tương đồng Cosine
$$\cos(\vec{A}, \vec{B}) = \frac{\vec{A}\cdot\vec{B}}{\|\vec{A}\|\,\|\vec{B}\|}$$
- Đo **góc** giữa 2 vector, không phụ thuộc độ dài bài (bài dài hay ngắn đều so sánh công bằng).
- Vì giá trị TF-IDF/BOW không âm nên Cosine nằm trong $[0, 1]$: **1** = giống hệt, **0** = không có từ chung.

Kết quả là ma trận đối xứng 6×6, đường chéo luôn bằng 1 (mỗi bài giống chính nó).

In [ ]:
def similarity(docs, method="tfidf"):
    stopwords = load_stopwords()
    corpus = [preprocess(d["text"], stopwords) for d in docs]     # văn bản đã làm sạch
    # Chọn cách xây dựng vector đặc trưng: TF-IDF hoặc BOW
    vectorizer = TfidfVectorizer() if method == "tfidf" else CountVectorizer()
    vectors = vectorizer.fit_transform(corpus)    # ma trận (số bài x số từ vựng)
    print(f"Số bài: {vectors.shape[0]} | Kích thước bộ từ vựng: {vectors.shape[1]}")
    names = [d["name"] for d in docs]
    # cosine_similarity tính Cosine cho mọi cặp hàng -> ma trận 6x6
    return pd.DataFrame(cosine_similarity(vectors), index=names, columns=names)

## 5. Bước 5 – Xếp hạng các cặp tin tức
- Với 6 bài có $C_6^2 = 15$ cặp khác nhau. `combinations(range(6), 2)` sinh các cặp `(a, b)` với `a < b`,
  tức chỉ lấy **nửa trên** ma trận (bỏ đường chéo và không lặp cặp `B–A`).
- Thêm cột **Quan hệ** (cùng / khác chuyên mục) để dễ kiểm tra kết quả có hợp lý không.
- Sắp xếp theo cột `Cosine` **giảm dần**, đánh số hạng từ 1.

In [ ]:
def rank_pairs(docs, sim):
    rows = []
    for a, b in combinations(range(len(docs)), 2):     # 15 cặp (a < b)
        da, db = docs[a], docs[b]
        rows.append({
            "Cặp": f"{da['name']} - {db['name']}",
            "Cosine": sim.iloc[a, b],                   # giá trị ở hàng a, cột b của ma trận
            "Quan hệ": "Cùng chuyên mục" if da["category"] == db["category"] else "Khác chuyên mục",
        })
    df = pd.DataFrame(rows).sort_values("Cosine", ascending=False).reset_index(drop=True)
    df.index += 1                                       # hạng bắt đầu từ 1
    return df

## 6. Hàm `main()` – tích hợp toàn bộ quy trình
Gọi lần lượt các bước: crawl → tiền xử lý + vector hoá + Cosine → xếp hạng → in kết quả.
Tham số `method`: `"tfidf"` (mặc định) hoặc `"bow"`.

In [ ]:
def main(method="tfidf"):
    print(f"=== Crawl dữ liệu realtime lúc {datetime.now():%d/%m/%Y %H:%M:%S} ===")
    docs = crawl_all()                                  # Bước 1

    print()
    sim = similarity(docs, method)                      # Bước 2, 3, 4
    print(f"\n=== Ma trận độ tương đồng Cosine ({method.upper()}) ===")
    print(sim.round(6))

    print("\n=== Các cặp tin tức theo độ tương đồng giảm dần ===")   # Bước 5
    print(rank_pairs(docs, sim).to_string(formatters={"Cosine": "{:.6f}".format}))

    print("\n=== Danh sách bài ===")
    for d in docs:
        print(f"{d['name']} [{d['category']}]: {d['title']}")
    return docs, sim

## 7. Chạy toàn bộ (1 lần) – dùng TF-IDF

In [ ]:
docs, sim = main("tfidf")

=== Crawl dữ liệu realtime lúc 22/10/2026 17:18:01 ===

[News1] Thể thao: https://vnexpress.net/rss/the-thao.rss


   + News11.txt | 22/10/2026 15:14 | Fabregas vượt trội Carrick thế nào về chiến thuật


   + News12.txt | 22/10/2026 14:50 | Sa thải Carrick có cứu được Man Utd?

[News2] Sức khỏe: https://vnexpress.net/rss/suc-khoe.rss


   + News21.txt | 22/10/2026 16:00 | Người bệnh đái tháo đường có nguy cơ suy thận?


   + News22.txt | 22/10/2026 16:00 | Từ Australia về Việt Nam tán sỏi thận

[News3] Kinh doanh: https://vnexpress.net/rss/kinh-doanh.rss


   + News31.txt | 22/10/2026 17:00 | Sun PhuQuoc Airways chuẩn bị mở đường bay đến Nga


   + News32.txt | 22/10/2026 15:55 | Phó thống đốc: 'Các ngân hàng cần giám sát chặt an toàn hệ thống'

Số bài: 6 | Kích thước bộ từ vựng: 886

=== Ma trận độ tương đồng Cosine (TFIDF) ===
          News11    News12    News21    News22    News31    News32
News11  1.000000  0.684172  0.019032  0.017729  0.022679  0.029435
News12  0.684172  1.000000  0.011355  0.008067  0.018559  0.029011
News21  0.019032  0.011355  1.000000  0.209645  0.005250  0.022529
News22  0.017729  0.008067  0.209645  1.000000  0.004865  0.003043
News31  0.022679  0.018559  0.005250  0.004865  1.000000  0.030363
News32  0.029435  0.029011  0.022529  0.003043  0.030363  1.000000

=== Các cặp tin tức theo độ tương đồng giảm dần ===
                Cặp   Cosine          Quan hệ
1   News11 - News12 0.684172  Cùng chuyên mục
2   News21 - News22 0.209645  Cùng chuyên mục
3   News31 - News32 0.030363  Cùng chuyên mục
4   News11 - News32 0.029435  Khác chuyên mục
5   News12 - News32 0.029011  Khác chuyên mục
6   News11 - 

## 8. (Tham khảo) So sánh với BOW – `CountVectorizer`
Dùng lại 6 bài vừa crawl (không crawl lại), chỉ đổi cách vector hoá để so sánh với kết quả TF-IDF.

In [ ]:
sim_bow = similarity(docs, "bow")
print(rank_pairs(docs, sim_bow).to_string(formatters={"Cosine": "{:.6f}".format}))

Số bài: 6 | Kích thước bộ từ vựng: 886
                Cặp   Cosine          Quan hệ
1   News11 - News12 0.728579  Cùng chuyên mục
2   News21 - News22 0.277513  Cùng chuyên mục
3   News31 - News32 0.056604  Cùng chuyên mục
4   News11 - News32 0.056450  Khác chuyên mục
5   News12 - News32 0.043317  Khác chuyên mục
6   News11 - News31 0.036505  Khác chuyên mục
7   News21 - News32 0.033475  Khác chuyên mục
8   News11 - News22 0.031248  Khác chuyên mục
9   News12 - News31 0.030817  Khác chuyên mục
10  News11 - News21 0.029145  Khác chuyên mục
11  News12 - News21 0.017496  Khác chuyên mục
12  News12 - News22 0.014320  Khác chuyên mục
13  News22 - News31 0.010177  Khác chuyên mục
14  News21 - News31 0.009685  Khác chuyên mục
15  News22 - News32 0.007577  Khác chuyên mục


## 9. Nhận xét kết luận
- Các cặp **cùng chuyên mục** thường đứng đầu bảng xếp hạng, nhất là khi 2 bài cùng nói về một sự kiện
  (ví dụ 2 bài thể thao cùng về Man Utd) → Cosine rất cao.
- Các cặp **khác chuyên mục** có Cosine gần 0 vì gần như không có từ khoá chung sau khi đã bỏ stopwords.
- Hai bài cùng chuyên mục nhưng khác chủ đề nhỏ (ví dụ hàng không và ngân hàng trong *Kinh doanh*)
  có thể chỉ cao hơn các cặp khác chuyên mục một chút.
- **BOW** thường cho Cosine cao hơn TF-IDF ở các cặp khác chủ đề, vì BOW không giảm trọng số những từ phổ biến;
  TF-IDF tách biệt các chủ đề rõ hơn.
- Do dữ liệu lấy **realtime**, mỗi lần chạy có thể ra bài khác nên các con số sẽ thay đổi; hãy đọc lại bảng output để nhận xét.